In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform

Sí, es viable, pero con dos matices importantes sobre los datos y sobre qué tipo de problema resolverías.

1. Datos disponibles

Million Playlist Dataset (MPD)

Ya no se descarga directamente de AIcrowd. La página indica que hay que pedir acceso a Spotify Research.
Existe una copia en Kaggle (himanshuwagh/spotify-million) y un registro en Zenodo (DOI 10.5281/zenodo.6425593). Revisa los términos de licencia antes de usarlos: la licencia original es solo para investigación no comercial y no permite redistribuir los datos.
No trae características de audio, solo metadatos (título, artista, álbum, duración y posición).

Características de audio (tempo, energy, valence, etc.)

Spotify eliminó los endpoints audio-features y audio-analysis para apps nuevas desde el 27 de noviembre de 2024. Solo los conservan las apps que ya tenían acceso ampliado antes de esa fecha.
Alternativas: datasets públicos con las características ya calculadas, como el archivo SpotifyAudioFeaturesApril2019.csv que usa el Space de Hugging Face que encontré antes (busca ese nombre en Kaggle y verifica la licencia), o servicios de terceros.
Otra opción es construir la distancia con el MPD mismo, usando la co-ocurrencia: dos canciones que aparecen juntas en muchas playlists están “cerca”.
2. Cómo conectarlo con TensorACO

Revisé el repositorio. El ejemplo del README construye el algoritmo con distances=jnp.load("problem/pcb442.npy"), es decir, recibe una matriz de distancias precalculada en lugar de coordenadas. Eso es el puente: no necesitas ciudades, puedes pasarle la matriz de distancias entre canciones.

Pipeline sugerido:

- Elige un conjunto de n canciones (una playlist o un pool grande del MPD).
- Normaliza sus características de audio.
- Calcula la matriz de distancias n × n (por ejemplo, euclidiana sobre las características, o 1 menos la similitud por co-ocurrencia).
- Guárdala como .npy y pásala a TensorACO y al problema TSP.
- El mejor “tour” es un orden de reproducción con transiciones suaves entre canciones.

3. Matices importantes
Resuelves ordenamiento, no continuación. El TSP decide en qué orden poner canciones que ya tienes, no cuáles agregar. Es un problema parecido pero distinto al del RecSys Challenge. El paper de Spotify dice que el orden exacto parece importar poco a los usuarios, así que conviene presentarlo como “transiciones suaves” y no como “mejor playlist”.
El TSP es un ciclo cerrado y una playlist no regresa al inicio. Una solución habitual es agregar una ciudad ficticia con distancia 0 a todas, o simplemente ignorar el tramo de regreso.
La ventaja de la tensorización se ve con n grande. Una playlist promedio tiene 66 canciones y ahí un ACO normal en CPU ya es rápido. Para comparar de forma interesante, usa un pool de miles de canciones y mide tiempo y costo de ruta contra un ACO clásico. Así replicas el espíritu del paper.
No verifiqué que el código acepte cualquier matriz (por ejemplo, si requiere que sea simétrica). Pruébalo primero con una matriz pequeña.
4. Requisitos técnicos del repositorio

Python 3.7 o superior, evox==0.8.0, jax==0.4.34 y jaxlib==0.4.34; CUDA es opcional, así que corre en CPU. La licencia es GPL-3.0.

Si quieres, puedo escribirte un script en Python que tome un CSV de canciones con características de audio, genere la matriz .npy y la corra con TensorACO. También puedo armarte un diseño de experimento para comparar contra un ACO clásico.

In [2]:
data = pd.read_csv("../Data/SpotifyAudioFeaturesApril2019.csv")

In [17]:
data.head(20)

,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.1180,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.3710,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.3820,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.6410,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.9280,0
5,Curbo,26Y1lX7ZOpw9Ql3gGAlqLK,Debauchery - Original mix,0.001150,0.810,124016,0.417,0.919,9,0.1060,-10.783,0,0.0793,120.025,4,0.8370,0
6,Bingo Play,5eIyK73BrxHLnly4F9PWqg,Grandma - Original mix,0.000539,0.819,132742,0.720,0.863,4,0.0727,-8.895,0,0.1510,124.003,4,0.9340,0
7,G Herbo,13Mf2ZBpfNkgWJowvM5hXh,Bon appétit,0.115000,0.885,181838,0.348,0.000,9,0.1070,-12.569,1,0.4510,142.111,4,0.1800,0
8,34 Feet,7BQaRTHk44DkMhIVNcXy2D,Among - Original mix,0.000058,0.740,124016,0.472,0.847,8,0.0959,-9.008,1,0.0551,120.034,4,0.6220,0
9,Chris Cooq,049RxG2laEl9U1PGYeIqLV,Hazard - Original mix,0.000081,0.813,132742,0.731,0.910,11,0.0727,-8.932,1,0.0697,124.031,4,0.9440,0


In [4]:
numeric_data = data.drop(['artist_name', 'track_id', 'track_name', 'popularity', 'mode'], axis =1)

In [5]:
numeric_data.mean()

acousticness             0.342500
danceability             0.581468
duration_ms         212633.124098
energy                   0.569196
instrumentalness         0.224018
key                      5.231894
liveness                 0.194886
loudness                -9.974006
speechiness              0.112015
tempo                  119.473353
time_signature           3.878986
valence                  0.439630
dtype: float64

In [6]:
norm_data = (numeric_data - numeric_data.mean())/numeric_data.std()

In [7]:
norm_data

,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,speechiness,tempo,time_signature,valence
0,-0.974076,0.849823,0.209004,-0.884309,-0.621707,-1.174645,-0.677780,0.350836,2.388743,2.800221,0.235251,-1.241435
1,-0.920321,1.391709,0.017595,-0.046851,-0.621707,0.768342,0.543206,0.414861,2.774821,1.310879,0.235251,-0.264901
2,-0.918585,0.113279,-0.598596,0.590846,-0.621707,1.045911,-0.670626,0.624048,-0.535805,-0.149450,0.235251,-0.222443
3,-0.905855,1.149701,-0.708474,0.037663,1.909321,-0.064367,-0.569274,-0.327609,-0.337135,0.117032,0.235251,0.777251
4,-0.990813,1.060264,-0.719557,0.855913,1.814962,0.490772,-0.963949,-0.046298,-0.369308,0.019020,0.235251,1.885019
...,...,...,...,...,...,...,...,...,...,...,...,...
130658,-0.971299,0.102757,-0.179401,0.890487,-0.621707,1.601050,-0.571063,0.733455,-0.641172,0.383813,0.235251,-0.581406
130659,1.665023,-1.023104,-0.146564,-1.230048,-0.621707,1.045911,-0.547812,-0.497984,-0.549478,-0.843324,0.235251,-0.361396
130660,-0.036166,0.713036,-0.596339,-0.142890,-0.621707,0.768342,-0.567486,0.248000,0.755951,1.243505,0.235251,0.410568
130661,-0.968059,1.002393,-0.264870,-0.227404,-0.621707,-0.341936,-0.380880,0.046453,0.064226,0.019087,0.235251,-1.405477


In [8]:
distances = pdist(norm_data.head(20).values, metric='euclidean')

In [9]:
distances

array([3.10248615, 5.16442844, 5.34810818, 6.16460688, 5.93569594,
       5.54561344, 3.2795195 , 5.4861685 , 6.47236122, 5.4924728 ,
       5.60959658, 6.34222601, 5.1362322 , 5.51042813, 4.97784152,
       5.74540824, 5.16273697, 5.46443773, 5.25511713, 4.13636759,
       4.65586693, 5.11354975, 4.79913317, 4.65153509, 1.88707554,
       4.57227728, 5.06168221, 4.90928958, 4.47136973, 5.01237803,
       4.89949956, 5.29078538, 4.80591353, 4.82467339, 5.31840792,
       4.42705758, 5.08293039, 3.32521109, 3.50218883, 3.58575268,
       3.81133877, 4.19401355, 2.86313512, 3.60124235, 4.02407315,
       3.04603627, 3.61227213, 3.71324696, 4.08686032, 3.7452564 ,
       3.14565024, 4.17632037, 3.05093543, 3.87173254, 1.57310217,
       1.50355494, 1.54300281, 4.67529429, 1.12150204, 2.18205209,
       2.00671968, 1.35097225, 1.99511965, 1.38157628, 1.5246629 ,
       1.25060669, 0.83086194, 2.71843044, 0.88030217, 1.20112295,
       1.65698092, 1.18179807, 5.34277686, 1.79637424, 1.20218

In [10]:
dist_matrix = squareform(distances)

In [11]:
dist_matrix

array([[0.        , 3.10248615, 5.16442844, 5.34810818, 6.16460688,
        5.93569594, 5.54561344, 3.2795195 , 5.4861685 , 6.47236122,
        5.4924728 , 5.60959658, 6.34222601, 5.1362322 , 5.51042813,
        4.97784152, 5.74540824, 5.16273697, 5.46443773, 5.25511713],
       [3.10248615, 0.        , 4.13636759, 4.65586693, 5.11354975,
        4.79913317, 4.65153509, 1.88707554, 4.57227728, 5.06168221,
        4.90928958, 4.47136973, 5.01237803, 4.89949956, 5.29078538,
        4.80591353, 4.82467339, 5.31840792, 4.42705758, 5.08293039],
       [5.16442844, 4.13636759, 0.        , 3.32521109, 3.50218883,
        3.58575268, 3.81133877, 4.19401355, 2.86313512, 3.60124235,
        4.02407315, 3.04603627, 3.61227213, 3.71324696, 4.08686032,
        3.7452564 , 3.14565024, 4.17632037, 3.05093543, 3.87173254],
       [5.34810818, 4.65586693, 3.32521109, 0.        , 1.57310217,
        1.50355494, 1.54300281, 4.67529429, 1.12150204, 2.18205209,
        2.00671968, 1.35097225, 1.99511965, 1

In [16]:
np.save('Distances', dist_matrix)

In [14]:
np.fill_diagonal(dist_matrix, 1e9)